# 1b · Train an anomaly model — SOLUTION

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("../../data/telemetry.csv", parse_dates=["timestamp"]).ffill().bfill()
features = ["habitat_temp_c", "o2_pct", "co2_ppm", "power_kw", "battery_pct", "dust_index", "water_l"]
X = df[features]

In [ ]:
model = IsolationForest(contamination=0.1, random_state=42)
df["pred"] = model.fit_predict(X)
df["pred_anomaly"] = (df["pred"] == -1).astype(int)
df["pred_anomaly"].value_counts()

In [ ]:
print(confusion_matrix(df["label"], df["pred_anomaly"]))
print(classification_report(df["label"], df["pred_anomaly"], digits=3))

In [ ]:
# Did it catch the big O2 drop?
df[df.o2_pct < 19.5][["timestamp", "o2_pct", "co2_ppm", "label", "pred_anomaly"]]

## Stretch: rolling features + a supervised comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(X, df["label"], test_size=0.3, random_state=0, stratify=df["label"])
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(classification_report(yte, clf.predict(Xte), digits=3))